# ARISE — 01 · Data Loading & Anomaly Injection

**Paper:** Duan, Xiao, Wang, Zhou, Liu, *"ARISE: Graph Anomaly Detection on Attributed Networks via Substructure Awareness"*, IEEE TNNLS 35(12), 2024.

ARISE detects **two kinds of anomalies** in an attributed network:

| type | description | how it's injected (Sec. V-A2) |
|------|-------------|-------------------------------|
| **Topology anomaly** | a *group* of unrelated nodes that are densely interconnected (an unusually dense substructure) | pick `clique_size` nodes, make them **fully connected**, repeat `num_cliques` times |
| **Attribute anomaly** | a node whose features differ strongly from its neighbourhood | for a node, sample `k_candidates` others and **swap in the most Euclidean-distant** feature vector |

Benchmarks have no ground-truth anomalies, so we *inject* them. This notebook loads a graph and injects both types.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # make the `arise` package importable
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


## 1. Load an attributed network
We use **Cora** (2708 papers, 1433-dim bag-of-words features, citation edges). If the download fails, a synthetic stochastic-block-model graph is used instead.

In [ ]:
from arise.data import load_dataset, load_synthetic

graph = load_dataset("cora")   # falls back to synthetic on download failure
print(graph.summary())

## 2. Inject anomalies (paper Section V-A2)
The per-dataset counts from the paper are stored in `PAPER_INJECTION`. For Cora: 5 cliques × 15 nodes = **75 topology anomalies**, and **75 attribute anomalies**.

In [ ]:
from arise.data import inject_anomalies, PAPER_INJECTION

cfg = PAPER_INJECTION["Cora"]
print("injection config:", cfg)
graph = inject_anomalies(graph, seed=42, **cfg)
print(graph.summary())

## 3. Inspect the injected anomalies

In [ ]:
import numpy as np
n = graph.num_nodes
print(f"nodes={n}  edges={graph.num_edges}  avg_degree={graph.avg_degree:.2f}")
print(f"topology anomalies: {graph.topo_mask.sum()}   attribute anomalies: {graph.attr_mask.sum()}")

# Degrees of topology anomalies vs normal nodes — cliques make them high-degree
deg = np.asarray(graph.adj.sum(1)).ravel()
print(f"mean degree  normal={deg[~graph.labels.astype(bool)].mean():.2f}  "
      f"topology-anom={deg[graph.topo_mask].mean():.2f}")

### Visualise one injected clique
Topology anomalies form fully-connected groups. We draw one clique and its neighbourhood.

In [ ]:
import networkx as nx
G = nx.from_scipy_sparse_array(graph.adj)

clique_nodes = np.where(graph.topo_mask)[0][:15]   # first injected clique
sub_nodes = set(clique_nodes)
for u in clique_nodes:
    sub_nodes.update(list(G.neighbors(u))[:2])
H = G.subgraph(sub_nodes)
colors = ["#ff5c5c" if x in set(clique_nodes) else "#6ea8ff" for x in H.nodes()]

plt.figure(figsize=(6,6))
nx.draw_networkx(H, node_color=colors, node_size=120, with_labels=False,
                 edge_color="#cccccc", pos=nx.spring_layout(H, seed=1))
plt.title("An injected topology anomaly (red = clique, blue = neighbours)")
plt.axis("off"); plt.show()

### Degree distribution

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(deg[~graph.labels.astype(bool)], bins=40, alpha=.6, label="normal", color="#6ea8ff")
plt.hist(deg[graph.topo_mask], bins=20, alpha=.8, label="topology anomaly", color="#ff5c5c")
plt.xlabel("degree"); plt.ylabel("count"); plt.legend(); plt.title("Degree distribution"); plt.show()

## Summary
- Loaded an attributed network and injected **topology** (dense cliques) and **attribute** (swapped features) anomalies exactly as in the paper.
- Topology anomalies are visibly higher-degree and tightly clustered — the signal ARISE's substructure module exploits.

➡️ Next: **02** trains the contrastive network that detects *attribute* anomalies (and produces the embeddings the topology module needs).